# Practice guide 04_01: static malware detection from PE headers and API fingerprints

**Goal.** Build the static detector yourself: the evaluation metrics of an imbalanced scanner (AUC, average
precision, precision/recall at an operating point), the benign-downsampling experiment, a two-stage
hierarchy over 7 malware families, and the malleable-field preview that class 04_03 exploits. Every `TODO`
cell must be completed; the `check` cells tell you whether you got it right.

Read `guide_04_01.pdf` first. Time: about 2 hours. Cells marked **(given)** are complete.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from scipy import sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

LIB = next(p for p in (Path("../../notebooks/03-malware"), Path("../../../notebooks/03-malware")) if p.exists())
sys.path.insert(0, str(LIB))  # student / solution folder
import malwlib as mw

## Part A. The PE-header data (given)

19,611 files x 77 numeric header fields, 74.4% malware — the *reverse* of a real scanner queue, which we
exploit in part C.

In [ ]:
X, y, cols = mw.load_pe()
print(f"PE data: {X.shape[0]:,} files x {X.shape[1]} fields, malware {y.mean():.2%}")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

**Question A1.** Why does `stratify=y` matter here less than in a 1%-malware problem — and when would it
matter at all?

## Part B. Ranking metrics from scratch

A scanner outputs a score; the analyst picks a threshold. Implement, from the sorted scores alone:

* `auc_score(y, s)`: the probability that a random malware file scores above a random benign one (handle
  ties with 0.5);
* `average_precision(y, s)`: precision summed at the rank of each positive, divided by the number of
  positives;
* `pr_at_threshold(y, s, t)`: precision and recall at threshold `t`.

No scikit-learn in your implementation — the check cells compare with it.

In [ ]:
def rank_matrix(y, s):
    """Sort by score descending; return the sorted labels and the scores."""
    o = np.argsort(s)[::-1]
    return y[o], s[o]


def auc_score(y, s):
    # TODO: sort once; pos = scores of y==1, neg of y==0. AUC = (#pairs pos>neg) + 0.5*(#ties) over (n_pos*n_neg).
    # Hint: np.searchsorted on the sorted neg array is the fast way; a double loop also passes the checks.
    raise NotImplementedError


def average_precision(y, s):
    # TODO: sort descending with rank_matrix; cumulative sum of positives / rank; take the value at every
    # positive and average.
    raise NotImplementedError


def pr_at_threshold(y, s, t):
    # TODO: predict s >= t; return (precision, recall) guarding against division by zero.
    raise NotImplementedError

In [ ]:
# check: a perfect and a random score, then the scikit-learn twins
from sklearn.metrics import average_precision_score, roc_auc_score

rng = np.random.default_rng(0)
yp = np.concatenate([np.ones(50), np.zeros(50)])
assert auc_score(yp, yp) == 1.0 and abs(auc_score(yp, rng.random(100)) - 0.5) < 0.15
s = rng.random(100) * np.where(yp, 1, 0.6)
assert abs(auc_score(yp, s) - roc_auc_score(yp, s)) < 1e-12
assert abs(average_precision(yp, s) - average_precision_score(yp, s)) < 1e-12
print(f"AUC {auc_score(yp, s):.3f}  AP {average_precision(yp, s):.3f}  metrics OK")

**Question B1.** A scanner flags a file when its score is above the threshold. With 74.4% malware in the
test set, what is the precision of a scanner that flags *every* file? And with the realistic 1% base rate?
Which quantity does the base rate enter directly?

## Part C. The detector and the imbalance experiment

Train the random forest (given), then downsample the benign *training* files to the fractions 1.0, 0.2, 0.05,
0.01 and measure, for each, the AUC, the AP and the precision at fixed recall 0.95. Fill the table.

In [ ]:
rf_full = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
p_full = rf_full.predict_proba(Xte)[:, 1]
print("full model:", mw.eval_bin(yte, p_full, "RF PE baseline: "))

In [ ]:
idx_b = np.where(ytr == 0)[0]
rowsC = {}
# TODO: for frac in [1.0, 0.2, 0.05, 0.01]:
#   downsample idx_b with rng.choice(..., size=int(len(idx_b)*frac), replace=False), keep all malware rows,
#   fit a fresh RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1) on the mixed rows,
#   score Xte, and store rowsC[frac] = (auc, ap, precision_at_recall_095).
# For the last column: pick the smallest threshold with recall >= 0.95 and report its precision.
for frac, (a, ap_, pr) in rowsC.items():
    print(f"benign frac {frac:5.2f}  AUC {a:.4f}  AP {ap_:.4f}  precision@recall.95 {pr:.4f}")

**Question C1.** Which column degrades as the benign fraction drops, and which barely moves? What does that
mean for a vendor who trains on a 1%-malware queue and advertises an AUC?

## Part D. API-function fingerprints: flat vs two-stage

The Windows API dataset counts 21,918 API functions per file over 7 classes. Loading it eagerly would need
tens of GB; `malwlib.load_api_sparse` reads 100-row UInt8 slices and keeps a stratified 20,000-file sample
(~6 MB CSR, peak RSS under 0.8 GB). **This cell takes a few minutes the first time; the demo cache makes it
instant afterwards.**

In [ ]:
CACHE = mw.ROOT / ".cache_api_sub"
if CACHE.with_suffix(".npz").exists():
    M, yf = sp.load_npz(CACHE.with_suffix(".npz")), np.load(CACHE.with_suffix(".npy"))
else:
    M, yf = mw.load_api_sparse()
    sp.save_npz(CACHE.with_suffix(".npz"), M)
    np.save(CACHE.with_suffix(".npy"), yf)
print(f"API matrix: {M.shape[0]:,} x {M.shape[1]:,}, density {M.nnz / (M.shape[0] * M.shape[1]):.4f}")

itr, ite = train_test_split(np.arange(len(yf)), test_size=0.25, stratify=yf, random_state=0)

Train the flat 7-class forest (given) and the two-stage pipeline: stage 1 binary benign-vs-malware, stage 2
family only on the files stage 1 calls malicious. Compare macro-F1 end to end.

In [ ]:
flat = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1).fit(M[itr], yf[itr])
pf = flat.predict(M[ite])
from sklearn.metrics import classification_report, f1_score

print(classification_report(yf[ite], pf, digits=3, zero_division=0))
flat_macro = f1_score(yf[ite], pf, average="macro")

In [ ]:
yb = (yf > 0).astype(int)
hier = {}
# TODO 1: stage 1 — RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1) on yb, predict on M[ite].
# TODO 2: stage 2 — train a forest on the TRUE malicious training rows only (yf[itr] > 0) with 6 classes;
#         at test time, apply it to the test rows where stage 1 predicts malicious (pred1 > 0).
# TODO 3: assemble the end-to-end prediction: 0 where stage 1 says benign, stage-2 family elsewhere;
#         store hier = (stage1_recall_malware, stage2_macro_f1_on_its_rows, end_to_end_macro_f1).
print(f"flat macro-F1 {flat_macro:.4f}")
print(f"hier: stage-1 recall {hier[0]:.4f}, stage-2 macro-F1 {hier[1]:.4f}, end-to-end macro-F1 {hier[2]:.4f}")

**Question D1.** The hierarchy wins (or loses) only a little macro-F1. Why does the *recall of stage 1* cap
the whole pipeline, and why can stage 2 never recover a file stage 1 missed?

## Part E. What the model looks at — and the preview of 04_03

Print the 8 most important header fields. `malwlib.MALLEABLE` lists the fields an attacker can rewrite
without breaking the program: count how many of the top-20 fields are malleable.

In [ ]:
# TODO: imp = rf_full.feature_importances_; print the top-8 (field name, importance);
#       compute n_mal = number of the top-20 fields whose column name is in mw.MALLEABLE; print it.

**Question E1.** Class 04_03 rewrites exactly these fields to the benign median and the detector folds. Two
defences come to mind that do **not** need retraining — name them and say why each is unsatisfying.

## Part G. A safe replicator — what the detector cannot even see

`malwlib.replicate` writes the source of a minimal self-copying program, generation by generation,
into a scratch directory. It carries no payload (it only writes the next generation and stops at a
hard cap), is never executed as a subprocess, and touches nothing outside `workdir`. Build 5
generations and read generation 0.

In [ ]:
import tempfile

workdir_g = Path(tempfile.mkdtemp(prefix="guide_04_01_replicator_"))
# TODO: paths_g = mw.replicate(workdir_g, cap=5); print the number of files and the text of generation 0
# (paths_g[0].read_text()).
raise NotImplementedError

**Question G1.** Could you build a 77-column row for this file the way `load_pe` does for the PE
dataset? Could a static detector trained on PE headers, or on API-import counts, return *any* score
for it? What specifically is missing — the schema, or the training data?

In [ ]:
import shutil

shutil.rmtree(workdir_g)  # clean up: this demo is confined to its own temp directory

# Reference solution

The solved notebook, with the answers to the questions and the numbers of a full run, is `solution/guide_04_01.ipynb`.